# Functions, scope, and modules

Functions divide a larger computation into reusable operations. A function groups a body of statements behind a name, accepts arguments through parameters, and returns a result to its caller. Modules organize related definitions into separate namespaces.

## Definition and invocation

A `def` statement creates a function object and binds its name. The indented body of an ordinary function runs when the function is called, not when it is defined. A function can have zero or more parameters.

An optional docstring is the first string literal in the body. It describes the operation, inputs, and result and is available through `__doc__` and `help()`.

A function object can also be assigned to another name. Calling that name invokes the same function.

In [1]:
def square(number):
    """Return number multiplied by itself."""
    return number * number


print(square(5))       # 25
operation = square
print(operation(3))    # 9
print(square.__doc__)  # Return number multiplied by itself.

25
9
Return number multiplied by itself.


## Parameters and arguments

Parameters are names in the function definition; arguments are values supplied by a call. Positional arguments bind according to order. Keyword arguments bind by parameter name, so their order can differ from the definition.

Required parameters need values, and a parameter cannot receive two values in one call. Default values make selected arguments optional. For ordinary positional-or-keyword parameters, required parameters precede parameters with defaults.

Default expressions are evaluated when the function is defined, not on every call.

In [2]:
def trip_cost(nights, nightly_rate=140, spending_money=0):
    """Return lodging and spending costs for a trip."""
    return nights * nightly_rate + spending_money


print(trip_cost(2))                                     # 280
print(trip_cost(2, 100, 30))                             # 230
print(trip_cost(spending_money=30, nights=2, nightly_rate=100))  # 230
try:
    trip_cost(2, nights=3)
except TypeError:
    print("nights received two values")

280
230
230
nights received two values


## Returning values and displaying output

`print()` displays values and returns `None`. `return` ends the current function call and supplies a value to the caller. The caller can assign, print, or pass that value to another operation; returning does not itself store a result permanently.

A function without an explicit `return`, or with a bare `return`, returns `None`. A function can contain several return statements in different branches. Execution leaves the function when a return is reached, so later ordinary statements on that path do not run.

In [3]:
def display_square(number):
    print(number * number)


def describe_sign(number):
    if number < 0:
        return "negative"
    if number == 0:
        return "zero"
    return "positive"


display_result = display_square(4)  # Prints 16.
print(display_result)               # None
computed_result = square(4)
print(computed_result)              # 16
print(describe_sign(-2), describe_sign(0), describe_sign(2))

16
None
16
negative zero positive


## Returning several components

Comma-separated values in a return statement are packed into one tuple. Tuple unpacking binds the components to separate names. The number of receiving names must match the number of components unless starred unpacking is used.

The example splits a nonnegative integer number of seconds into hours, minutes, and seconds.

In [4]:
def split_seconds(total_seconds):
    """Split a nonnegative integer duration into hours, minutes, seconds."""
    hours = total_seconds // 3600
    remainder = total_seconds % 3600
    minutes = remainder // 60
    seconds = remainder % 60
    return hours, minutes, seconds


parts = split_seconds(4000)
print(parts)  # (1, 6, 40)
hours, minutes, seconds = parts
print(hours, minutes, seconds)  # 1 6 40

(1, 6, 40)
1 6 40


## Calls, frames, and local names

Each ordinary function call creates an execution frame with its parameter bindings and local state. Arguments are bound before the body executes. A local assignment changes a local binding, not a same-named variable in the caller.

The frame stops executing when the call finishes. Objects can outlive the call if references to them remain, including returned objects and values retained by nested functions.

Arguments refer to objects rather than automatic copies. Rebinding a parameter is different from mutating an object passed as an argument.

In [5]:
value = 10


def increment(value):
    value += 1
    return value


print(increment(value))  # 11
print(value)             # 10


def add_label(labels):
    labels.append("new")  # Changes the shared list object.


labels = ["existing"]
add_label(labels)
print(labels)  # ['existing', 'new']

11
10
['existing', 'new']


## Name lookup and enclosing scopes

In ordinary function code, unqualified names are resolved through local, enclosing function, global module, and built-in scopes, often abbreviated LEGB. A nested function can read names from its enclosing function.

Assignment normally makes a name local to the current function. Reading that local name before it has a value raises `UnboundLocalError`, even if a global name with the same spelling exists. An assignment in an inner function does not automatically update an enclosing binding.

In [6]:
def apply_rate(amount):
    rate = 0.1

    def fee(value):
        return value * rate

    return fee(amount)


print(apply_rate(50))  # 5.0

counter = 0


def invalid_increment():
    counter += 1  # Assignment makes counter local, but it has no local value yet.


try:
    invalid_increment()
except UnboundLocalError:
    print("Local counter has no value")

5.0
Local counter has no value


## `global` and `nonlocal`

`global` makes assignments target a name in the defining module's global namespace. `nonlocal` targets an existing binding in an enclosing function scope. Reading these names does not require either declaration; rebinding them does.

Mutating an existing object is separate from rebinding its name. A function can mutate a referenced list without a `global` declaration. Passing values in and returning results often makes dependencies clearer than modifying global state.

In [7]:
counter = 0


def increment_global():
    global counter
    counter += 1


increment_global()
print(counter)  # 1


def enclosing_count():
    count = 0

    def increment_inner():
        nonlocal count
        count += 1

    increment_inner()
    increment_inner()
    return count


print(enclosing_count())  # 2

1
2


## Mutable default arguments

A mutable default object is shared across calls that omit that argument. A list default can therefore retain changes made during earlier calls.

A `None` default allows a fresh list to be created when no list is supplied. Passing an existing list still shares that object with the function.

In [8]:
def collect(value, items=None):
    """Append value to a supplied list, or to a fresh list when omitted."""
    if items is None:
        items = []
    items.append(value)
    return items


print(collect("a"))  # ['a']
print(collect("b"))  # ['b']
shared = ["start"]
print(collect("end", shared))  # ['start', 'end']
print(shared)                 # ['start', 'end']

['a']
['b']
['start', 'end']
['start', 'end']


## Modules and imports

A module groups definitions in its own namespace; a Python source file can define a module. A library may contain several modules or packages. Splitting a larger program into focused functions and modules separates responsibilities and supports reuse.

`import math` binds the module name, and attribute access with a dot selects a member such as `math.sqrt`. `from math import sqrt` binds that member directly. `as` supplies an alias.

Importing a module normally executes its top-level code on first import in a process. Subsequent imports reuse the cached module. Importing is therefore more than simply copying function definitions.

In [9]:
import math
from math import sqrt as square_root

print(math.sqrt(81))     # 9.0
print(square_root(81))   # 9.0
print(round(math.pi, 3)) # 3.142

9.0
9.0
3.142


## Inspection and documentation

`dir()` lists attribute names for interactive inspection; not every attribute is a function. `help()` displays documentation for a function, type, or module. A function's `__doc__` exposes its docstring directly.

```python
help(square)
help(math)
```

Built-ins such as `len`, `sum`, and `max` are available without imports. Reusing those names for variables hides the built-in in that scope.

In [10]:
print("sqrt" in dir(math))  # True
print(square.__doc__)
print(len([2, 4, 6]), sum([2, 4, 6]), max([2, 4, 6]))  # 3 12 6

True
Return number multiplied by itself.
3 12 6


Related concepts: [branching](03_branching.ipynb), [lists and shared objects](06_lists.ipynb), and the [module entry-point examples](if_name_main/).